In [ ]:
import pandas as pd
import re
import pytz
from collections import Counter
from openpyxl.styles import Font, Alignment, Border, Side

# ============================================================
# CONFIG
# ============================================================
INPUT_FILE  = r"C:\Users\ADMIN\Downloads\Expedia VN_Schedule_WB0601.xlsx"
OUTPUT_FILE = r"C:\Users\ADMIN\Downloads\Agent_and_Sup_Output.xlsx"
# ============================================================

LOB_MAP = {
    "Lodging":             "Lodging",
    "Lodging_Nesting":     "Lodging",
    "Non_Lodging":         "Non-Loding",
    "Non_Lodging_Nesting": "Non-Loding",
    "Support_LG_Nesting":  "Lodging",
    "Support_NL_Nesting":  "Non-Loding",
}

REST_VALUES = {"OFF", "CO", "AL", "LWP"}
VN_TZ  = pytz.timezone("Asia/Ho_Chi_Minh")
PST_TZ = pytz.timezone("America/Los_Angeles")

def is_shift(v):
    return bool(re.match(r"^\d{4}-\d{4}$", str(v).strip()))

def vn_to_pst(vn_shift: str, date_ref) -> str:
    """Convert ca VN (HHMM-HHMM) -> PST/PDT string dùng pytz, tự xử lý DST."""
    s = vn_shift.strip()
    try:
        start_str, end_str = s[:4], s[5:]
        date_ref = pd.to_datetime(date_ref).date()

        vn_start = VN_TZ.localize(pd.to_datetime(f"{date_ref} {start_str[:2]}:{start_str[2:]}:00"))
        vn_end   = VN_TZ.localize(pd.to_datetime(f"{date_ref} {end_str[:2]}:{end_str[2:]}:00"))

        pst_start = vn_start.astimezone(PST_TZ)
        pst_end   = vn_end.astimezone(PST_TZ)

        start_time = pst_start.strftime("%I:%M:%S %p").lstrip("0")
        end_time   = pst_end.strftime("%I:%M:%S %p").lstrip("0")
        return f"{start_time} to {end_time}"
    except Exception:
        return s

def dominant_shift(row, date_cols):
    shifts = [str(row.get(c, "")).strip() for c in date_cols if is_shift(str(row.get(c, "")))]
    return Counter(shifts).most_common(1)[0][0] if shifts else None

def get_shift_pst(row, date_cols, date_label):
    main = dominant_shift(row, date_cols)
    if main:
        # Dùng ngày đầu tuần làm date_ref cho pytz
        return vn_to_pst(main, date_cols[0])
    return f"Off on {date_label}"

def rest_days(row, date_cols):
    return ", ".join(
        col.strftime("%A") for col in date_cols
        if str(row.get(col, "")).strip().upper() in REST_VALUES
    )

def has_no_shift_whole_week(row, date_cols):
    return not any(is_shift(str(row.get(c, ""))) for c in date_cols)

def title_case(name):
    return " ".join(w.capitalize() for w in str(name).split())

# ── Read ──────────────────────────────────────────────────────
xl  = pd.ExcelFile(INPUT_FILE)
df  = pd.read_excel(xl, sheet_name=xl.sheet_names[0], header=0)

date_cols = [c for c in df.columns if hasattr(c, "strftime")]
if not date_cols:
    print("Columns:", df.columns.tolist())
    raise ValueError("Không tìm thấy cột ngày.")

date_label = date_cols[0].strftime("%#d %B")  # Windows: %#d | Linux/Mac: %-d

print(f"Sheet      : {xl.sheet_names[0]}")
print(f"Date cols  : {[c.strftime('%Y-%m-%d') for c in date_cols]}")
print(f"Week of    : {date_cols[0].strftime('%Y-%m-%d')} → {date_cols[-1].strftime('%Y-%m-%d')}")

# ── Filter ────────────────────────────────────────────────────
df_work = df[
    df["Email"].notna() &
    df["Email"].astype(str).str.contains("@", na=False) &
    ~df["LOB"].astype(str).str.contains("Flex", case=False, na=False)
].copy()

before = len(df_work)
df_work = df_work[~df_work.apply(lambda r: has_no_shift_whole_week(r, date_cols), axis=1)]
print(f"Agents     : {len(df_work)}  (removed {before - len(df_work)} rows with no shift whole week)")

# ── Transform ─────────────────────────────────────────────────
rows = []
for _, row in df_work.iterrows():
    email   = str(row["Email"]).strip()
    lob_raw = str(row.get("LOB", "Lodging")).strip()

    rows.append({
        "Agent Name":        title_case(str(row["Employee Name"])),
        "OKTA ID":           email,
        "Salesforce ID":     email,
        "Channel Type":      "Chat",
        "Location":          "Concentrix (Ho Chi Minh)",
        "Shift (PST Hours)": get_shift_pst(row, date_cols, date_label),
        "Rest Day":          rest_days(row, date_cols),
        "Role":              "Agent",
        "Access Level":      "Tier 1 Agent",
        "LOB":               LOB_MAP.get(lob_raw, "Lodging"),
    })

out = pd.DataFrame(rows).sort_values("Agent Name").reset_index(drop=True)
off_ct = out["Shift (PST Hours)"].str.startswith("Off on").sum()
print(f"Working    : {len(out) - off_ct}  |  Off: {off_ct}")

# ── Write Excel ───────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    out.to_excel(writer, sheet_name="Sheet1", index=False)
    ws = writer.sheets["Sheet1"]

    thin    = Side(style="thin")
    bdr     = Border(left=thin, right=thin, top=thin, bottom=thin)
    h_font  = Font(name="Arial", bold=True, size=10)
    d_font  = Font(name="Arial", size=10)
    c_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    l_align = Alignment(horizontal="left",   vertical="center", wrap_text=True)

    for col_letter, w in zip("ABCDEFGHIJ", [30,40,40,14,30,30,45,10,16,14]):
        ws.column_dimensions[col_letter].width = w

    for cell in ws[1]:
        cell.font, cell.alignment, cell.border = h_font, c_align, bdr

    for row_cells in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row_cells:
            cell.font, cell.alignment, cell.border = d_font, l_align, bdr

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

print(f"✓ Saved → {OUTPUT_FILE}")

Sheet      : Sheet2
Date cols  : ['2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04', '2026-06-05', '2026-06-06', '2026-06-07']
Week of    : 2026-06-01 → 2026-06-07
Agents     : 141  (removed 6 rows with no shift whole week)
Working    : 141  |  Off: 0
✓ Saved → C:\Users\ADMIN\Downloads\Agent_and_Sup_Output.xlsx
